<a href="https://colab.research.google.com/github/dsaenzd04/IA_PROGRAMACION/blob/main/modelo_trigramas_didactico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini-modelo de Lenguaje con Trigramas

## Cual es el reto

En este proyecto vamos a construir un **modelo que genera texto** mirando **las dos palabras anteriores** para predecir la siguiente. Es la evolucion natural del modelo de bigramas: pasamos de mirar **una** palabra atras a mirar **dos**.

Este es el mismo principio que usan los grandes modelos de lenguaje: predecir la siguiente palabra a partir del contexto anterior. La diferencia es que ellos miran miles de palabras atras con redes neuronales; nosotros miraremos dos palabras con estadistica simple.

## Recorrido del proyecto

1. **Preparar el corpus** - limpiar el texto y separarlo en oraciones.
2. **Construir el vocabulario** - asignar un numero a cada palabra.
3. **Contar trigramas** - que trios de palabras aparecen juntos.
4. **Calcular probabilidades** - que probabilidad hay de que tras `(w1, w2)` venga `w3`.
5. **Muestrear** - elegir la siguiente palabra respetando esas probabilidades.
6. **Generar frases** - encadenar palabras desde dos semillas.
7. **Ida y vuelta** - codificar, decodificar y comprobar que no se pierde informacion.
8. **Guardar y cargar el modelo** - para no tener que reentrenar cada vez.
9. **Probar todo** con pruebas unitarias.

## Que necesitas saber antes de empezar

- Variables, listas y diccionarios en Python.
- Bucles `for` y condicionales `if`.
- Funciones con `def` y `return`.
- Haber entendido el modelo de bigramas (o, al menos, la idea de contar pares de palabras).

## Que cambia respecto a bigramas

Resumen de las diferencias clave:

| Aspecto | Bigramas | Trigramas |
|---|---|---|
| Que se cuenta | Pares `(w1, w2)` | Trios `(w1, w2, w3)` |
| Contexto | Una palabra | Dos palabras |
| Probabilidad | `P(w2 \| w1)` | `P(w3 \| w1, w2)` |
| Denominador | `contextos[w1]` | `contextos[(w1, w2)]` |
| Semilla para generar | 1 palabra | 2 palabras |
| Coherencia del texto | Baja | Mejor |
| Dispersion (datos escasos) | Alta | Mucho mas alta |

---
# Concepto clave: que es un trigrama

Un **trigrama** es una **secuencia de tres palabras consecutivas** dentro de una oracion.

## Ejemplo

Tomemos la oracion:

> *"la inteligencia artificial es el futuro"*

Sus trigramas son todas las ternas consecutivas:

| Trigrama | Se lee como |
|---|---|
| `('la', 'inteligencia', 'artificial')` | tras *la inteligencia* viene *artificial* |
| `('inteligencia', 'artificial', 'es')` | tras *inteligencia artificial* viene *es* |
| `('artificial', 'es', 'el')` | tras *artificial es* viene *el* |
| `('es', 'el', 'futuro')` | tras *es el* viene *futuro* |

Una oracion de **N** palabras tiene **N - 2** trigramas.

## Por que usar trigramas

Con bigramas, el modelo solo mira **la ultima palabra**. Por eso puede generar cosas como:

> *"el futuro de la tecnologia son rapidos"*

Con trigramas, mira **dos palabras**. Eso le da mas contexto y las frases suelen ser mas coherentes:

> *"el futuro de la tecnologia es prometedor"*

## El precio a pagar: dispersion

Cuantas mas palabras mira el modelo, **mas combinaciones posibles** hay, y por tanto **menos veces aparece cada combinacion** en el corpus. Esto se llama **dispersion de datos** (data sparsity).

Por eso, en trigramas el **suavizado de Laplace** es todavia mas importante que en bigramas: sin el, casi todas las probabilidades serian 0.

---
# Concepto clave: que es el contexto

En un modelo de trigramas, el **contexto** de una palabra son **las dos palabras que la preceden**.

## Ejemplo

En la oracion `'la inteligencia artificial es'`:

- El contexto de `'artificial'` es `('la', 'inteligencia')`.
- El contexto de `'es'` es `('inteligencia', 'artificial')`.
- El contexto de `'la'` es **nada** (es la primera palabra).
- El contexto de `'inteligencia'` es **casi nada** (solo tiene una palabra antes).

Por eso, cuando contamos trigramas, **empezamos desde la tercera palabra**: las dos primeras solo pueden ser contexto, no final de un trigrama.

Cuando contamos trigramas, tambien contamos cuantas veces aparece cada **par de palabras como contexto**. Ese numero sera el **denominador** al calcular probabilidades.

## Ejemplo con numeros

Supongamos que en el corpus aparece:

- El par `('de', 'la')` como contexto **8 veces**.
- El trigrama `('de', 'la', 'tecnologia')` **3 veces**.
- El trigrama `('de', 'la', 'vida')` **2 veces**.

Eso significa que, de las 8 veces que aparecio el par `('de', 'la')` como inicio de un trigrama, 3 veces le siguio `'tecnologia'`, 2 veces `'vida'`, y las otras 3 les siguieron otras palabras.

---
# Concepto clave: como se calcula la probabilidad

La pregunta que queremos responder ahora es:

> *Dadas las palabras `w1` y `w2`, cual es la probabilidad de que la siguiente palabra sea `w3`?*

Se escribe `P(w3 | w1, w2)` y significa *"probabilidad de `w3` dados `w1` y `w2`"*.

## Formula basica

```
                       veces que aparece (w1, w2, w3)
P(w3 | w1, w2) = ------------------------------------------
                       veces que aparece (w1, w2) como contexto
```

Es decir: **casos favorables dividido por casos totales**, igual que en bigramas, pero ahora los casos se refieren a pares y ternas.

## Ejemplo

Con los numeros anteriores:

- `('de', 'la', 'tecnologia')` aparece **3 veces**.
- `('de', 'la')` como contexto aparece **8 veces**.

```
P('tecnologia' | 'de', 'la') = 3 / 8 = 0.375  -> 37.5% de probabilidad
P('vida'       | 'de', 'la') = 2 / 8 = 0.250  -> 25.0% de probabilidad
```

## El problema de la probabilidad cero

Con trigramas, la **dispersion** es enorme: muchisimos trigramas posibles nunca aparecen en el corpus. Por ejemplo `('de', 'la', 'pinguino')` tendria probabilidad 0 sin suavizado.

Eso anularia muchisimas transiciones y haria que el modelo fuera muy fragil.

## Solucion: suavizado de Laplace (add-one)

Sumamos **1** al numerador y **V** (tamano del vocabulario) al denominador:

```
                       trigramas[(w1, w2, w3)] + 1
P(w3 | w1, w2) = --------------------------------------
                       contextos[(w1, w2)] + V
```

Con esto:

- Un trigrama visto 3 veces aporta un poco mas.
- Un trigrama nunca visto aporta un poquito (no cero).
- La suma de todas las probabilidades sigue siendo 1.

## Nota importante sobre `<UNK>`

Igual que en bigramas, `<UNK>` **forma parte del vocabulario** y **tambien recibe probabilidad** con Laplace. La distribucion debe sumar 1 sobre **todas** las palabras del vocabulario, incluida `<UNK>`.

Al muestrear (Etapa 4) filtraremos `<UNK>` para no escribirlo en la frase generada. Pero en la distribucion si aparece.

---
# Caja de herramientas: funciones base

Antes de construir el modelo, cargamos las funciones del tokenizador que ya conocemos. Las usaremos en todas las etapas:

- `limpiar_texto` -> minusculas y sin puntuacion.
- `tokenizar` -> dividir en palabras.
- `construir_vocabulario_codificar` -> palabra a ID.
- `construir_vocabulario_decodificar` -> ID a palabra.
- `codificar_frase` / `decodificar_secuencia` -> conversion entre texto y numeros.

Ejecuta la siguiente celda tal cual: solo carga funciones, no hace nada mas.

In [ ]:
def limpiar_texto(texto, caracteres_a_quitar=None):
    """Pasa a minusculas y elimina puntuacion basica."""
    if caracteres_a_quitar is None:
        caracteres_a_quitar = ['.', ',', '!', '¿', '?', '¡', '(', ')', '"']
    texto_limpio = texto.lower()
    for caracter in caracteres_a_quitar:
        texto_limpio = texto_limpio.replace(caracter, '')
    return texto_limpio


def tokenizar(texto_limpio):
    """Divide el texto limpio en una lista de palabras."""
    return texto_limpio.split()


def construir_vocabulario_codificar(palabras, token_unk='<UNK>'):
    """Crea el diccionario palabra -> ID. El ID 0 es para <UNK>."""
    vocab = {token_unk: 0}
    for palabra in palabras:
        if palabra not in vocab:
            vocab[palabra] = len(vocab)
    return vocab


def construir_vocabulario_decodificar(vocab_codificar):
    """Invierte el vocabulario para obtener ID -> palabra."""
    return {id_num: palabra for palabra, id_num in vocab_codificar.items()}


def codificar_frase(frase_nueva, vocab_codificar):
    """Convierte una frase en IDs. Palabras desconocidas -> 0 (<UNK>)."""
    palabras = tokenizar(limpiar_texto(frase_nueva))
    return [vocab_codificar.get(p, 0) for p in palabras]


def decodificar_id(codigo, vocab_decodificar):
    """Devuelve la palabra de un ID o <UNK> si no existe."""
    return vocab_decodificar.get(codigo, '<UNK>')


def decodificar_secuencia(secuencia_numerica, vocab_decodificar):
    """Convierte una lista de IDs en una cadena de texto."""
    palabras = [decodificar_id(c, vocab_decodificar) for c in secuencia_numerica]
    return ' '.join(palabras)


def mostrar_vocab_aprendido(vocab):
    """Imprime el vocabulario de forma legible."""
    for clave, valor in vocab.items():
        print(f"{clave} : {valor}")


print("Funciones base cargadas correctamente.")

Funciones base cargadas correctamente.


---
# Etapa 1 - Preparar el corpus y el vocabulario

## Que queremos conseguir en esta etapa

Tener el texto listo para analizarlo:

1. **Limpiar** cada oracion (minusculas, sin puntuacion).
2. **Separar** el corpus en oraciones individuales.
3. **Construir** un vocabulario: cada palabra unica recibe un ID.

## Por que guardamos las oraciones por separado

Igual que en bigramas, es **imprescindible** mantener las oraciones separadas. Si unimos todo el corpus, aparecerian trigramas falsos que cruzan el final de una oracion con el inicio de la siguiente.

**Regla de oro:** los trigramas solo existen **dentro** de los limites de cada oracion.

In [ ]:
def preparar_corpus(texto_crudo):
    """Devuelve una lista de oraciones, cada una como lista de tokens limpios."""
    oraciones = []
    for linea in texto_crudo.strip().split('\n'):
        limpia = limpiar_texto(linea)
        tokens = tokenizar(limpia)
        if tokens:
            oraciones.append(tokens)
    return oraciones


def recolectar_vocabulario(oraciones):
    """Construye el vocabulario (palabra -> ID) a partir de una lista de oraciones."""
    todas_las_palabras = [p for oracion in oraciones for p in oracion]
    return construir_vocabulario_codificar(todas_las_palabras)


# --- Prueba rapida ---
corpus_demo = """La Inteligencia Artificial es el futuro de la tecnologia.
La Inteligencia Artificial es el presente de la tecnologia.
Los diccionarios en Python son extremadamente rapidos.
Las tablas hash permiten busquedas en tiempo constante.
La programacion en Python es divertida y poderosa.
El futuro de la tecnologia es prometedor.
La tecnologia avanza rapidamente cada dia."""

oraciones = preparar_corpus(corpus_demo)
vocab_cod = recolectar_vocabulario(oraciones)
vocab_dec = construir_vocabulario_decodificar(vocab_cod)

print("Numero de oraciones:", len(oraciones))
for i, o in enumerate(oraciones, 1):
    print(f"   Oracion {i}: {o}")

print("\nTamano del vocabulario:", len(vocab_cod))
print("Palabras del vocabulario:")
mostrar_vocab_aprendido(vocab_cod)

Numero de oraciones: 7
   Oracion 1: ['la', 'inteligencia', 'artificial', 'es', 'el', 'futuro', 'de', 'la', 'tecnologia']
   Oracion 2: ['la', 'inteligencia', 'artificial', 'es', 'el', 'presente', 'de', 'la', 'tecnologia']
   Oracion 3: ['los', 'diccionarios', 'en', 'python', 'son', 'extremadamente', 'rapidos']
   Oracion 4: ['las', 'tablas', 'hash', 'permiten', 'busquedas', 'en', 'tiempo', 'constante']
   Oracion 5: ['la', 'programacion', 'en', 'python', 'es', 'divertida', 'y', 'poderosa']
   Oracion 6: ['el', 'futuro', 'de', 'la', 'tecnologia', 'es', 'prometedor']
   Oracion 7: ['la', 'tecnologia', 'avanza', 'rapidamente', 'cada', 'dia']

Tamano del vocabulario: 33
Palabras del vocabulario:
<UNK> : 0
la : 1
inteligencia : 2
artificial : 3
es : 4
el : 5
futuro : 6
de : 7
tecnologia : 8
presente : 9
los : 10
diccionarios : 11
en : 12
python : 13
son : 14
extremadamente : 15
rapidos : 16
las : 17
tablas : 18
hash : 19
permiten : 20
busquedas : 21
tiempo : 22
constante : 23
programacion 

## Preguntas de control (Etapa 1)

1. Por que guardamos las oraciones por separado?
2. Por que usamos un corpus con frases repetidas en parte (por ejemplo, dos veces "La Inteligencia Artificial es")?
3. Cuantos trigramas tiene una oracion de 8 palabras?

> Respuestas:
> 1. Para no crear trigramas falsos entre el final de una oracion y el inicio de la siguiente.
> 2. Porque con trigramas hay mucha dispersion. Repetir frases ayuda a que el modelo tenga ejemplos claros de continuaciones.
> 3. N - 2 = 6 trigramas.

---
# Etapa 2 - Contar trigramas y contextos

## Que queremos conseguir

Recorrer todas las oraciones y llenar dos diccionarios:

### 1. `trigramas`
Cuantas veces aparece cada terna `(w1, w2, w3)`.

```
{
  ('la', 'inteligencia', 'artificial'): 2,
  ('de', 'la', 'tecnologia'): 3,
  ...
}
```

### 2. `contextos`
Cuantas veces aparece cada **par** `(w1, w2)` como inicio de un trigrama.

```
{
  ('la', 'inteligencia'): 2,
  ('de', 'la'): 3,
  ...
}
```

Este segundo diccionario sera el **denominador** al calcular probabilidades.

## Como lo hacemos sin `Counter`

Con un diccionario normal y el metodo `.get(clave, 0)`:

```python
trigramas[terna] = trigramas.get(terna, 0) + 1
contextos[par] = contextos.get(par, 0) + 1
```

## Detalle importante: el rango del bucle

Fijate en que ahora el bucle va de `0` a `len(tokens) - 3` **inclusive**, es decir, `range(len(tokens) - 2)`. Asi nos aseguramos de que siempre haya **tres** palabras disponibles.

In [ ]:
def contar_trigramas(oraciones):
    """Devuelve un dict con las ternas (w1, w2, w3) y su frecuencia."""
    trigramas = {}
    for tokens in oraciones:
        for i in range(len(tokens) - 2):
            terna = (tokens[i], tokens[i + 1], tokens[i + 2])
            trigramas[terna] = trigramas.get(terna, 0) + 1
    return trigramas


def contar_contextos(oraciones):
    """Cuenta cuantas veces aparece cada par (w1, w2) como inicio de un trigrama."""
    contextos = {}
    for tokens in oraciones:
        for i in range(len(tokens) - 2):
            par = (tokens[i], tokens[i + 1])
            contextos[par] = contextos.get(par, 0) + 1
    return contextos


def top_n(diccionario, n):
    """Devuelve los n pares (clave, valor) mas frecuentes."""
    return sorted(diccionario.items(), key=lambda kv: kv[1], reverse=True)[:n]


# --- Prueba rapida ---
trigramas = contar_trigramas(oraciones)
contextos = contar_contextos(oraciones)

print("Trigramas mas frecuentes:")
for terna, freq in top_n(trigramas, 5):
    print(f"   {terna} -> {freq}")

print("\nContextos (pares) mas frecuentes:")
for par, freq in top_n(contextos, 5):
    print(f"   {par} -> {freq}")

print("\nComprobacion manual:")
print(f"   El trigrama ('de', 'la', 'tecnologia') aparece {trigramas.get(('de','la','tecnologia'), 0)} vez/veces")
print(f"   El par ('de', 'la') es contexto {contextos.get(('de','la'), 0)} vez/veces")
print(f"   Por tanto P('tecnologia' | 'de', 'la') = {trigramas.get(('de','la','tecnologia'),0)} / {contextos.get(('de','la'),0)}")

Trigramas mas frecuentes:
   ('de', 'la', 'tecnologia') -> 3
   ('la', 'inteligencia', 'artificial') -> 2
   ('inteligencia', 'artificial', 'es') -> 2
   ('artificial', 'es', 'el') -> 2
   ('el', 'futuro', 'de') -> 2

Contextos (pares) mas frecuentes:
   ('de', 'la') -> 3
   ('la', 'inteligencia') -> 2
   ('inteligencia', 'artificial') -> 2
   ('artificial', 'es') -> 2
   ('es', 'el') -> 2

Comprobacion manual:
   El trigrama ('de', 'la', 'tecnologia') aparece 3 vez/veces
   El par ('de', 'la') es contexto 3 vez/veces
   Por tanto P('tecnologia' | 'de', 'la') = 3 / 3


## Preguntas de control (Etapa 2)

1. Cual es la diferencia entre `trigramas` y `contextos`?
2. Si `contextos[('de','la')] = 3`, que significa ese 3?
3. Por que el bucle va de `0` a `len(tokens) - 3` inclusive?
4. Por que los contextos de un trigrama son **pares** y no palabras sueltas?

> Respuestas:
> 1. `trigramas` cuenta ternas; `contextos` cuenta cuantas veces cada par inicia un trigrama.
> 2. Que el par `('de', 'la')` aparece 3 veces como las dos primeras palabras de un trigrama.
> 3. Porque necesitamos tener tres palabras disponibles: `i`, `i+1` e `i+2`. Cuando `i = len(tokens) - 3`, accedemos a las tres ultimas palabras.
> 4. Porque la probabilidad es `P(w3 | w1, w2)`: el contexto son las **dos** palabras anteriores, no una.

---
# Etapa 3 - Probabilidades con suavizado de Laplace

## La formula

Recordemos:

```
                       trigramas[(w1, w2, w3)] + 1
P(w3 | w1, w2) = --------------------------------------
                       contextos[(w1, w2)] + V
```

- `+1` en el numerador -> suavizado (nada tiene probabilidad 0).
- `+V` en el denominador -> compensa los +1 y mantiene la suma en 1.
- **V** = numero de palabras del vocabulario.

## Ejemplo numerico

Supongamos:

- `trigramas[('de', 'la', 'tecnologia')] = 3`
- `contextos[('de', 'la')] = 3`
- `V = 40`

Entonces:

```
P('tecnologia' | 'de', 'la') = (3 + 1) / (3 + 40) = 4 / 43 = 0.093
```

Y para una palabra nunca vista tras ese contexto:

```
P('pinguino' | 'de', 'la') = (0 + 1) / (3 + 40) = 1 / 43 = 0.023
```

Pequenito, pero **mayor que cero**. Eso es exactamente lo que queremos.

## Nota importante sobre `<UNK>`

Igual que en bigramas, `<UNK>` forma parte del vocabulario y tambien recibe probabilidad. La distribucion debe sumar 1 sobre **todas** las palabras del vocabulario, incluida `<UNK>`. Al muestrear (Etapa 4) lo filtraremos para no escribirlo.

## Las funciones que vamos a escribir

- `probabilidad_trigrama(w1, w2, w3, ...)` -> devuelve un numero entre 0 y 1.
- `distribucion_siguiente(w1, w2, ...)` -> devuelve un diccionario `{palabra: probabilidad}` con **todas** las palabras del vocabulario como posibles continuaciones de `(w1, w2)`. La suma es 1.

In [ ]:
def probabilidad_trigrama(w1, w2, w3, trigramas, contextos, vocab_size):
    """P(w3 | w1, w2) con suavizado de Laplace."""
    conteo_terna = trigramas.get((w1, w2, w3), 0)
    conteo_contexto = contextos.get((w1, w2), 0)
    return (conteo_terna + 1) / (conteo_contexto + vocab_size)


def distribucion_siguiente(w1, w2, trigramas, contextos, vocab_decodificar):
    """Devuelve un dict {palabra: probabilidad} de TODAS las palabras del vocabulario
    como posibles continuaciones de (w1, w2)."""
    vocab_size = len(vocab_decodificar)
    distribucion = {}
    for palabra in vocab_decodificar.values():
        distribucion[palabra] = probabilidad_trigrama(w1, w2, palabra, trigramas, contextos, vocab_size)
    return distribucion


# --- Prueba rapida ---
V = len(vocab_cod)
print("Tamano del vocabulario V =", V)

print("\nProbabilidades de algunas continuaciones de ('de', 'la'):")
for w in ['tecnologia', 'vida', 'python', 'hash']:
    p = probabilidad_trigrama('de', 'la', w, trigramas, contextos, V)
    print(f"   P('{w}' | 'de', 'la') = {p:.4f}")

dist = distribucion_siguiente('de', 'la', trigramas, contextos, vocab_dec)
print("\nTop 5 continuaciones de ('de', 'la'):")
for palabra, p in top_n(dist, 5):
    print(f"   {palabra:15s} {p:.4f}")

suma = sum(dist.values())
print(f"\nSuma de todas las probabilidades: {suma:.6f}  (debe ser = 1)")
print(f"La distribucion contiene <UNK>? {'<UNK>' in dist}")

Tamano del vocabulario V = 33

Probabilidades de algunas continuaciones de ('de', 'la'):
   P('tecnologia' | 'de', 'la') = 0.1111
   P('vida' | 'de', 'la') = 0.0278
   P('python' | 'de', 'la') = 0.0278
   P('hash' | 'de', 'la') = 0.0278

Top 5 continuaciones de ('de', 'la'):
   tecnologia      0.1111
   <UNK>           0.0278
   la              0.0278
   inteligencia    0.0278
   artificial      0.0278

Suma de todas las probabilidades: 1.000000  (debe ser = 1)
La distribucion contiene <UNK>? True


## Preguntas de control (Etapa 3)

1. Por que sumamos 1 al numerador?
2. Por que sumamos `V` al denominador?
3. Si `V` fuera 100 y `contextos[('de','la')] = 3`, cual seria la probabilidad de una palabra nunca vista tras `('de','la')`?
4. Por que la distribucion debe incluir `<UNK>`?

> Respuestas:
> 1. Para que ninguna transicion tenga probabilidad 0.
> 2. Para compensar los +1 y mantener la suma de probabilidades igual a 1.
> 3. `1 / (3 + 100) = 1/103 = 0.0097`.
> 4. Porque `<UNK>` es una palabra mas del vocabulario y la distribucion debe sumar 1 sobre todas las palabras.

---
# Etapa 4 - Muestrear la siguiente palabra

## La herramienta: `random.choices`

Igual que en bigramas, usamos:

```python
random.choices(palabras, weights=probabilidades, k=1)[0]
```

## Y `<UNK>`?

La distribucion incluye `<UNK>` (para sumar 1). Pero **no queremos escribir `<UNK>` en una frase generada**. Al muestrear lo **filtramos** de la lista de candidatas.

## Por que fijamos `random.seed(...)`

Para que los resultados sean **reproducibles** y las pruebas unitarias sean deterministas.

In [ ]:
import random

def muestrear_siguiente_palabra(w1, w2, trigramas, contextos, vocab_decodificar):
    """Dadas dos palabras (w1, w2), devuelve la siguiente muestreada segun P(· | w1, w2),
    excluyendo <UNK> de las candidatas."""
    distribucion = distribucion_siguiente(w1, w2, trigramas, contextos, vocab_decodificar)
    candidatas = {p: pr for p, pr in distribucion.items() if p != '<UNK>'}
    palabras = list(candidatas.keys())
    pesos    = list(candidatas.values())
    return random.choices(palabras, weights=pesos, k=1)[0]


# --- Prueba rapida ---
random.seed(42)
print("10 muestras de P(· | 'de', 'la'):")
for _ in range(10):
    print("   ", muestrear_siguiente_palabra('de', 'la', trigramas, contextos, vocab_dec))

# Comprobacion estadistica
random.seed(42)
N = 5000
conteos = {}
for _ in range(N):
    w = muestrear_siguiente_palabra('de', 'la', trigramas, contextos, vocab_dec)
    conteos[w] = conteos.get(w, 0) + 1

print(f"\nFrecuencia empirica vs teorica ({N} muestras):")
print(f"   {'palabra':15s} {'empirica':>10s} {'teorica':>10s}")
for palabra, freq in top_n(conteos, 5):
    empirica = freq / N
    teorica = probabilidad_trigrama('de', 'la', palabra, trigramas, contextos, V)
    print(f"   {palabra:15s} {empirica:10.4f} {teorica:10.4f}")

10 muestras de P(· | 'de', 'la'):
    permiten
    la
    tecnologia
    tecnologia
    constante
    busquedas
    avanza
    es
    en
    inteligencia

Frecuencia empirica vs teorica (5000 muestras):
   palabra           empirica    teorica
   tecnologia          0.1186     0.1111
   permiten            0.0314     0.0278
   es                  0.0312     0.0278
   poderosa            0.0310     0.0278
   extremadamente      0.0308     0.0278


## Preguntas de control (Etapa 4)

1. Que diferencia hay entre **elegir la mas probable** y **muestrear**?
2. Para que sirve `random.seed`?
3. Por que la distribucion incluye `<UNK>` pero el muestreo no lo devuelve?

> Respuestas:
> 1. Elegir la mas probable siempre da la misma palabra (determinista). Muestrear da variedad, respetando las probabilidades.
> 2. Para que los resultados sean reproducibles.
> 3. La distribucion debe sumar 1, y para ello incluye `<UNK>`. Pero al generar texto filtramos `<UNK>` para no escribir esa etiqueta tecnica.

---
# Etapa 5 - Generar una frase completa

## La gran diferencia con bigramas

Para predecir la siguiente palabra necesitamos **dos palabras de contexto**. Por tanto, la semilla ya no es una palabra suelta, sino **un par de palabras**:

```
semilla = ['de', 'la']
```

A partir de ahi:

1. Miramos las dos ultimas palabras de la frase.
2. Muestreamos la siguiente palabra dado ese par.
3. La anadimos y repetimos.

```
['de', 'la']  ->  muestrear  ->  ['de', 'la', 'tecnologia']
              ->  muestrear  ->  ['de', 'la', 'tecnologia', 'es']
              ->  ...
```

## Criterio de parada

Usamos un **numero fijo** de palabras (`n_palabras`). Como minimo debe ser 2, porque la semilla ya ocupa dos posiciones.

## Por que las frases mejoran respecto a bigramas

Porque el modelo tiene mas contexto. Ya no basta con que `'tecnologia'` suela seguir a `'la'`; tiene que seguir a `('de', 'la')` concretamente. Eso filtra muchas continuaciones absurdas.

In [ ]:
def generar_frase(semilla, trigramas, contextos, vocab_decodificar, n_palabras=10):
    """Genera una frase de n_palabras a partir de una semilla de DOS palabras.
    La semilla debe ser una lista o tupla de dos palabras existentes en el vocabulario."""
    if len(semilla) != 2:
        raise ValueError("La semilla debe tener exactamente 2 palabras")
    if n_palabras < 2:
        raise ValueError("n_palabras debe ser al menos 2")

    frase = list(semilla)
    for _ in range(n_palabras - 2):
        w1, w2 = frase[-2], frase[-1]
        siguiente = muestrear_siguiente_palabra(w1, w2, trigramas, contextos, vocab_decodificar)
        frase.append(siguiente)
    return frase


# --- Prueba rapida ---
random.seed(42)
print("Tres frases generadas desde ('de', 'la'):")
for i in range(3):
    frase = generar_frase(['de', 'la'], trigramas, contextos, vocab_dec, n_palabras=10)
    print(f"   Frase {i+1}: {' '.join(frase)}")

print("\nFrases desde distintas semillas:")
random.seed(1)
semillas = [
    ['la', 'inteligencia'],
    ['en', 'python'],
    ['de', 'la'],
    ['el', 'futuro'],
    ['la', 'tecnologia'],
]
for s in semillas:
    frase = generar_frase(s, trigramas, contextos, vocab_dec, n_palabras=9)
    print(f"   [{s[0]:10s} {s[1]:14s}] {' '.join(frase)}")

Tres frases generadas desde ('de', 'la'):
   Frase 1: de la permiten la presente tecnologia programacion tiempo avanza artificial
   Frase 2: de la en la de las la de busquedas tablas
   Frase 3: de la tecnologia permiten y la y constante diccionarios el

Frases desde distintas semillas:
   [la         inteligencia  ] la inteligencia artificial poderosa divertida presente rapidos extremadamente busquedas
   [en         python        ] en python divertida es la poderosa son divertida la
   [de         la            ] de la python programacion tecnologia cada avanza la la
   [el         futuro        ] el futuro las cada python de son la tecnologia
   [la         tecnologia    ] la tecnologia son rapidos tecnologia tecnologia tecnologia extremadamente los


## Preguntas de control (Etapa 5)

1. Por que la semilla ahora tiene **dos** palabras?
2. Que pasa si la semilla no existe en el vocabulario?
3. Por que las frases de trigramas suelen ser mas coherentes que las de bigramas?

> Respuestas:
> 1. Porque la probabilidad es `P(w3 | w1, w2)`: el contexto son dos palabras.
> 2. `contextos.get((w1,w2), 0)` devuelve 0, se aplica Laplace y todas las palabras tienen probabilidad parecida. El modelo genera ruido, pero no falla.
> 3. Porque mira mas contexto y descarta continuaciones que solo serian probables con una palabra de contexto.

---
# Etapa 6 - Ida y vuelta: codificar, decodificar, comparar

## Por que hacemos esto

Unimos el **tokenizador** (palabras y numeros) con el **modelo de trigramas**:

```
tokens  ->  codificar  ->  IDs  ->  decodificar  ->  tokens
```

Como la frase la genero el propio modelo, todas sus palabras existen en el vocabulario. Por tanto la ida y vuelta debe ser **identica**.

## El caso interesante: palabra desconocida

Si metemos a proposito una palabra que no esta en el vocabulario:

- En modo estricto se convierte en `0` -> `<UNK>` al decodificar.
- El sistema **no colapsa**.

In [ ]:
def ida_y_vuelta(frase_tokens, vocab_cod, vocab_dec):
    """Codifica una lista de tokens y la vuelve a decodificar."""
    ids = [vocab_cod.get(p, 0) for p in frase_tokens]
    reconstruida = decodificar_secuencia(ids, vocab_dec)
    return ids, reconstruida.split()


def comparar_frases(original, reconstruida):
    """Devuelve un dict con el resultado de la comparacion."""
    return {
        "original": original,
        "reconstruida": reconstruida,
        "longitudes_iguales": len(original) == len(reconstruida),
        "identicas": original == reconstruida,
        "diferencias": [
            (i, a, b)
            for i, (a, b) in enumerate(zip(original, reconstruida))
            if a != b
        ],
    }


# --- Prueba 1: frase generada por el modelo ---
random.seed(42)
frase = generar_frase(['de', 'la'], trigramas, contextos, vocab_dec, n_palabras=10)
ids, reconstruida = ida_y_vuelta(frase, vocab_cod, vocab_dec)
resultado = comparar_frases(frase, reconstruida)

print("Frase generada   :", " ".join(frase))
print("IDs              :", ids)
print("Reconstruida     :", " ".join(reconstruida))
print("Identicas?       ", resultado["identicas"])
print("Diferencias      ", resultado["diferencias"])

# --- Prueba 2: frase con palabra desconocida a proposito ---
print("\n--- Prueba con palabra desconocida ---")
frase_desconocida = ['de', 'la', 'tecnologia', 'cuantica']
ids2, reconstruida2 = ida_y_vuelta(frase_desconocida, vocab_cod, vocab_dec)
resultado2 = comparar_frases(frase_desconocida, reconstruida2)

print("Original     :", " ".join(frase_desconocida))
print("IDs          :", ids2)
print("Reconstruida :", " ".join(reconstruida2))
print("Identicas?   ", resultado2["identicas"])
print("Diferencias  ", resultado2["diferencias"])

Frase generada   : de la permiten la presente tecnologia programacion tiempo avanza artificial
IDs              : [7, 1, 20, 1, 9, 8, 24, 22, 29, 3]
Reconstruida     : de la permiten la presente tecnologia programacion tiempo avanza artificial
Identicas?        True
Diferencias       []

--- Prueba con palabra desconocida ---
Original     : de la tecnologia cuantica
IDs          : [7, 1, 8, 0]
Reconstruida : de la tecnologia <UNK>
Identicas?    False
Diferencias   [(3, 'cuantica', '<UNK>')]


## Preguntas de control (Etapa 6)

1. Por que la ida y vuelta es exacta para frases generadas por el modelo?
2. Que ID recibe una palabra desconocida en modo estricto?
3. Que aparece al decodificar ese ID?

> Respuestas:
> 1. Porque todas las palabras generadas salen del vocabulario.
> 2. El ID 0 (reservado para `<UNK>`).
> 3. La etiqueta `<UNK>`.

---
# Etapa 7 - Guardar y cargar el modelo en JSON

## Que guardamos

Cuatro cosas:

1. **Version** del formato.
2. **`vocab_cod`** (palabra a ID).
3. **`trigramas`** (ternas a conteo).
4. **`contextos`** (pares a conteo).

**No** guardamos `vocab_dec` porque se reconstruye invirtiendo `vocab_cod`.

## El problema de las tuplas como claves

JSON **solo admite claves de tipo texto**. Pero nuestras claves de `trigramas` son **ternas** y las de `contextos` son **pares**. No se pueden guardar directamente.

**Solucion**: usar un separador `" ||| "` para unir las palabras:

```
('de', 'la', 'tecnologia')  ->  "de ||| la ||| tecnologia"
('de', 'la')                ->  "de ||| la"
```

Al cargar, deshacemos la conversion con `.split(SEPARADOR)` y `tuple(...)`.

## Funcion generica

Fijate que la misma funcion sirve para pares y ternas: `tuple(clave.split(SEPARADOR))` devuelve una tupla con 2 o 3 elementos, segun cuantos haya.

In [ ]:
import json

SEPARADOR = " ||| "


def tuplas_a_json(diccionario):
    """Convierte dict {(w1, ..., wn): n}  ->  dict {'w1 ||| ... ||| wn': n}."""
    return {SEPARADOR.join(clave): valor for clave, valor in diccionario.items()}


def json_a_tuplas(diccionario_json):
    """Convierte dict {'w1 ||| ... ||| wn': n}  ->  dict {(w1, ..., wn): n}."""
    return {tuple(clave.split(SEPARADOR)): valor for clave, valor in diccionario_json.items()}


def guardar_modelo(ruta, vocab_cod, trigramas, contextos):
    """Guarda el modelo completo en un archivo JSON."""
    modelo = {
        "version": 1,
        "tipo": "trigramas",
        "vocab_cod": vocab_cod,
        "trigramas": tuplas_a_json(trigramas),
        "contextos": tuplas_a_json(contextos),
    }
    with open(ruta, "w", encoding="utf-8") as f:
        json.dump(modelo, f, ensure_ascii=False, indent=2)


def cargar_modelo(ruta):
    """Carga el modelo desde un archivo JSON y reconstruye vocab_dec."""
    with open(ruta, "r", encoding="utf-8") as f:
        modelo = json.load(f)
    vocab_cod = modelo["vocab_cod"]
    trigramas = json_a_tuplas(modelo["trigramas"])
    contextos = json_a_tuplas(modelo["contextos"])
    vocab_dec = construir_vocabulario_decodificar(vocab_cod)
    return vocab_cod, vocab_dec, trigramas, contextos


# --- Prueba de ida y vuelta ---
guardar_modelo("modelo_trigramas.json", vocab_cod, trigramas, contextos)
vc2, vd2, tg2, cx2 = cargar_modelo("modelo_trigramas.json")

print("Vocabulario igual: ", vocab_cod == vc2)
print("Trigramas iguales: ", trigramas  == tg2)
print("Contextos iguales: ", contextos == cx2)
print("Tamano vocab:      ", len(vc2))
print("N trigramas:       ", len(tg2))

# Generamos con el modelo cargado: deberia dar la misma frase
random.seed(42)
frase_cargada = generar_frase(['de', 'la'], tg2, cx2, vd2, n_palabras=10)
print("\nFrase con el modelo cargado:")
print("   ", " ".join(frase_cargada))

print("\n--- Primeras lineas del JSON ---")
with open("modelo_trigramas.json", "r", encoding="utf-8") as f:
    print(f.read()[:700])

Vocabulario igual:  True
Trigramas iguales:  True
Contextos iguales:  True
Tamano vocab:       33
N trigramas:        33

Frase con el modelo cargado:
    de la permiten la presente tecnologia programacion tiempo avanza artificial

--- Primeras lineas del JSON ---
{
  "version": 1,
  "tipo": "trigramas",
  "vocab_cod": {
    "<UNK>": 0,
    "la": 1,
    "inteligencia": 2,
    "artificial": 3,
    "es": 4,
    "el": 5,
    "futuro": 6,
    "de": 7,
    "tecnologia": 8,
    "presente": 9,
    "los": 10,
    "diccionarios": 11,
    "en": 12,
    "python": 13,
    "son": 14,
    "extremadamente": 15,
    "rapidos": 16,
    "las": 17,
    "tablas": 18,
    "hash": 19,
    "permiten": 20,
    "busquedas": 21,
    "tiempo": 22,
    "constante": 23,
    "programacion": 24,
    "divertida": 25,
    "y": 26,
    "poderosa": 27,
    "prometedor": 28,
    "avanza": 29,
    "rapidamente": 30,
    "cada": 31,
    "dia": 32
  },
  "trigramas": {
    "la ||| intelige


## Preguntas de control (Etapa 7)

1. Por que no guardamos `vocab_dec` en el JSON?
2. Cuantas palabras hay en las claves del JSON de `trigramas`? Y en las de `contextos`?
3. Por que la misma funcion `tuplas_a_json` sirve para pares y ternas?

> Respuestas:
> 1. Porque se reconstruye invirtiendo `vocab_cod`. Ahorra espacio y evita inconsistencias.
> 2. 3 palabras en `trigramas`, 2 palabras en `contextos`.
> 3. Porque solo une los elementos de la tupla con el separador, sin importar cuantos sean.

---
# Etapa 8 - Pruebas unitarias

## Que vamos a probar

Al menos una prueba por etapa:

| Etapa | Que comprobamos |
|---|---|
| 1 | El corpus se separa bien y el vocabulario tiene `<UNK>=0` |
| 2 | Los trigramas no cruzan oraciones y se cuentan bien |
| 3 | Las probabilidades suman 1 y nunca son 0 |
| 4 | Las palabras muestreadas estan en el vocabulario y nunca son `<UNK>` |
| 5 | La frase generada empieza por la semilla y tiene N palabras |
| 6 | La ida y vuelta es identica y las palabras desconocidas van a `<UNK>` |
| 7 | Guardar y cargar conserva los conteos |

In [ ]:
print("=== PRUEBAS UNITARIAS DEL MODELO DE TRIGRAMAS ===\n")

# ---------- Etapa 1 ----------
print("Etapa 1: preparar corpus y vocabulario")

o = preparar_corpus("hola mundo\n\nadios mundo")
assert len(o) == 2, f"Esperaba 2 oraciones, obtuve {len(o)}"
assert o[0] == ['hola', 'mundo']

v = recolectar_vocabulario([['a', 'b', 'a'], ['b', 'c']])
assert v['<UNK>'] == 0
assert v['a'] == 1 and v['b'] == 2 and v['c'] == 3

vd = construir_vocabulario_decodificar(v)
for palabra, idn in v.items():
    assert vd[idn] == palabra

print("   OK - Todas las pruebas de la Etapa 1 pasaron\n")

# ---------- Etapa 2 ----------
print("Etapa 2: contar trigramas y contextos")

# Oracion de 5 palabras: a b c b c -> trigramas: (a,b,c), (b,c,b), (c,b,c)
tg = contar_trigramas([['a', 'b', 'c', 'b', 'c']])
assert tg[('a', 'b', 'c')] == 1, f"Esperaba 1, obtuve {tg[('a', 'b', 'c')]}"
assert tg[('b', 'c', 'b')] == 1
assert tg[('c', 'b', 'c')] == 1

# Una oracion de 2 palabras no genera ningun trigrama
assert contar_trigramas([['a', 'b']]) == {}

# Los trigramas NO deben cruzar oraciones
tg2 = contar_trigramas([['a', 'b', 'c'], ['d', 'e', 'f']])
assert ('c', 'd', 'e') not in tg2, "Los trigramas no deben cruzar oraciones"

# Contextos: cada par que inicia un trigrama, contado UNA vez por trigrama
cx = contar_contextos([['a', 'b', 'c', 'b', 'c']])
assert cx[('a', 'b')] == 1
assert cx[('b', 'c')] == 1
assert cx[('c', 'b')] == 1

print("   OK - Todas las pruebas de la Etapa 2 pasaron\n")

# ---------- Etapa 3 ----------
print("Etapa 3: probabilidades y suavizado de Laplace")

V_prueba = len(vocab_cod)
dist = distribucion_siguiente('de', 'la', trigramas, contextos, vocab_dec)
suma = sum(dist.values())
assert abs(suma - 1.0) < 1e-6, f"La distribucion no suma 1, suma {suma}"

# La distribucion debe incluir <UNK>
assert '<UNK>' in dist, "La distribucion debe incluir <UNK>"

p_rara = probabilidad_trigrama('de', 'la', 'palabraquenoexiste', trigramas, contextos, V_prueba)
assert p_rara > 0, "Laplace debe evitar el 0"

print("   OK - Todas las pruebas de la Etapa 3 pasaron\n")

# ---------- Etapa 4 ----------
print("Etapa 4: muestreo")

random.seed(0)
for _ in range(50):
    w = muestrear_siguiente_palabra('de', 'la', trigramas, contextos, vocab_dec)
    assert w in vocab_cod, f"Palabra fuera del vocabulario: {w}"

random.seed(0)
for _ in range(200):
    w = muestrear_siguiente_palabra('de', 'la', trigramas, contextos, vocab_dec)
    assert w != '<UNK>', "El muestreo no debe devolver <UNK>"

print("   OK - Todas las pruebas de la Etapa 4 pasaron\n")

# ---------- Etapa 5 ----------
print("Etapa 5: generar frase")

random.seed(42)
fr = generar_frase(['de', 'la'], trigramas, contextos, vocab_dec, n_palabras=10)
assert len(fr) == 10, f"Esperaba 10 palabras, obtuve {len(fr)}"
assert fr[0] == 'de' and fr[1] == 'la', "La frase debe empezar por la semilla"

# Semilla invalida lanza error
try:
    generar_frase(['solo_una'], trigramas, contextos, vocab_dec, n_palabras=5)
    assert False, "Deberia haber lanzado ValueError"
except ValueError:
    pass

print("   OK - Todas las pruebas de la Etapa 5 pasaron\n")

# ---------- Etapa 6 ----------
print("Etapa 6: ida y vuelta")

frase_ok = ['de', 'la', 'tecnologia', 'es', 'prometedor']
ids, rec = ida_y_vuelta(frase_ok, vocab_cod, vocab_dec)
assert frase_ok == rec, f"Esperaba {frase_ok}, obtuve {rec}"

frase_mal = ['de', 'la', 'tecnologia', 'cuantica']
ids2, rec2 = ida_y_vuelta(frase_mal, vocab_cod, vocab_dec)
assert ids2[3] == 0, "Palabra desconocida debe ser 0"
assert rec2[3] == '<UNK>', "ID 0 debe decodificarse como <UNK>"

print("   OK - Todas las pruebas de la Etapa 6 pasaron\n")

# ---------- Etapa 7 ----------
print("Etapa 7: guardar y cargar modelo")

guardar_modelo("modelo_test_tri.json", vocab_cod, trigramas, contextos)
vc_t, vd_t, tg_t, cx_t = cargar_modelo("modelo_test_tri.json")
assert vc_t == vocab_cod, "Vocabulario distinto tras cargar"
assert tg_t == trigramas, "Trigramas distintos tras cargar"
assert cx_t == contextos, "Contextos distintos tras cargar"

print("   OK - Todas las pruebas de la Etapa 7 pasaron\n")

print("=== TODAS LAS PRUEBAS PASARON ===")

=== PRUEBAS UNITARIAS DEL MODELO DE TRIGRAMAS ===

Etapa 1: preparar corpus y vocabulario
   OK - Todas las pruebas de la Etapa 1 pasaron

Etapa 2: contar trigramas y contextos
   OK - Todas las pruebas de la Etapa 2 pasaron

Etapa 3: probabilidades y suavizado de Laplace
   OK - Todas las pruebas de la Etapa 3 pasaron

Etapa 4: muestreo
   OK - Todas las pruebas de la Etapa 4 pasaron

Etapa 5: generar frase
   OK - Todas las pruebas de la Etapa 5 pasaron

Etapa 6: ida y vuelta
   OK - Todas las pruebas de la Etapa 6 pasaron

Etapa 7: guardar y cargar modelo
   OK - Todas las pruebas de la Etapa 7 pasaron

=== TODAS LAS PRUEBAS PASARON ===


---
# Demostracion final

Recorrido completo del sistema en cinco pasos:

1. Entrenamos el modelo con un corpus.
2. Generamos una frase desde una semilla de dos palabras.
3. La codificamos, la decodificamos y comprobamos que es identica.
4. Guardamos el modelo en JSON.
5. Lo cargamos desde cero y volvemos a generar la **misma** frase.

In [ ]:
# 1) Entrenar
corpus_final = """La Inteligencia Artificial es el futuro de la tecnologia.
La Inteligencia Artificial es el presente de la tecnologia.
Los diccionarios en Python son extremadamente rapidos.
Las tablas hash permiten busquedas en tiempo constante.
La programacion en Python es divertida y poderosa.
El futuro de la tecnologia es prometedor.
La tecnologia avanza rapidamente cada dia.
La Inteligencia Artificial transforma el mundo."""

import os
archivo ="1378-Jaldun-Muqaddimah.txt"
corpus=[]
if os.path.exists(archivo):
   with open(archivo,"r",encoding="utf-8") as f:
    for linea in f:
      corpus.append(linea.strip())
   corpus_final="\n".join(corpus)

oraciones_f = preparar_corpus(corpus_final)
vocab_cod_f = recolectar_vocabulario(oraciones_f)
vocab_dec_f = construir_vocabulario_decodificar(vocab_cod_f)
trigramas_f = contar_trigramas(oraciones_f)
contextos_f = contar_contextos(oraciones_f)

print(f"Vocabulario: {len(vocab_cod_f)} palabras")
print(f"Trigramas:   {len(trigramas_f)} ternas distintas")
print(f"Contextos:   {len(contextos_f)} pares distintos")

# 2) Generar
random.seed(7)
frase_gen = generar_frase(['la', 'inteligencia'], trigramas_f, contextos_f, vocab_dec_f, n_palabras=12)
print("\nFrase generada:")
print("   ", " ".join(frase_gen))

# 3) Ida y vuelta
ids_gen, frase_rec = ida_y_vuelta(frase_gen, vocab_cod_f, vocab_dec_f)
print("\nIDs:", ids_gen)
print("Reconstruida:", " ".join(frase_rec))
print("Identicas?", frase_gen == frase_rec)

# 4) Guardar
guardar_modelo("modelo_final_trigramas.json", vocab_cod_f, trigramas_f, contextos_f)
print("\nModelo guardado en 'modelo_final_trigramas.json'")

# 5) Cargar desde cero y regenerar
vocab_cod_c, vocab_dec_c, trigramas_c, contextos_c = cargar_modelo("modelo_final_trigramas.json")
random.seed(7)
frase_gen_c = generar_frase(['la', 'inteligencia'], trigramas_c, contextos_c, vocab_dec_c, n_palabras=12)
print("\nFrase desde el modelo cargado:")
print("   ", " ".join(frase_gen_c))
print("\nMisma frase que antes?", frase_gen == frase_gen_c)

Vocabulario: 44530 palabras
Trigramas:   342769 ternas distintas
Contextos:   189714 pares distintos

Frase generada:
    la inteligencia andar; raciones seres[9] podríamos ilícito” acabaron anales colocas políticos disolución

IDs: [6, 1073, 14399, 6718, 28986, 3226, 23863, 16284, 2583, 22596, 1670, 19310]
Reconstruida: la inteligencia andar; raciones seres[9] podríamos ilícito” acabaron anales colocas políticos disolución
Identicas? True

Modelo guardado en 'modelo_final_trigramas.json'

Frase desde el modelo cargado:
    la inteligencia andar; raciones seres[9] podríamos ilícito” acabaron anales colocas políticos disolución

Misma frase que antes? True


---
# Resumen final: que hemos aprendido

## Conceptos

- **Trigrama**: secuencia de tres palabras consecutivas `(w1, w2, w3)`.
- **Contexto**: las **dos** palabras que preceden, `(w1, w2)`.
- **Probabilidad de transicion**: `P(w3 | w1, w2)` = cuantas veces aparece la terna dividido por cuantas veces aparece el par como contexto.
- **Suavizado de Laplace**: sumar 1 al numerador y `V` al denominador para que nada tenga probabilidad 0.
- **Muestreo**: elegir al azar respetando las probabilidades.
- **Dispersion de datos**: a mayor contexto, mas combinaciones posibles y menos ejemplos de cada una.

## Diferencias clave respecto a bigramas

| Aspecto | Bigramas | Trigramas |
|---|---|---|
| Clave de conteo | `(w1, w2)` | `(w1, w2, w3)` |
| Clave de contexto | `w1` | `(w1, w2)` |
| Semilla | 1 palabra | 2 palabras |
| Bucle de conteo | `range(len(t) - 1)` | `range(len(t) - 2)` |
| Coherencia | Baja | Mejor |
| Dispersion | Alta | Mucho mas alta |

## Habilidades de programacion

- Diccionarios con **tuplas de 2 y 3 elementos** como claves.
- `.get(clave, por_defecto)` para evitar `KeyError`.
- Comprensiones de listas y diccionarios.
- `random.choices` con pesos y `random.seed`.
- `json.dump` y `json.load` para persistencia.
- `assert` y `try/except` para pruebas unitarias.

## Limitaciones del modelo

- Solo mira **dos** palabras atras.
- Mucha dispersion: la mayoria de trigramas posibles nunca aparecen.
- No entiende significado, solo estadistica.

## Y ahora que

Ideas para seguir ampliando:

1. **Cuatrigramas**: mirar tres palabras atras.
2. **Interpolacion**: combinar bigramas y trigramas con pesos.
3. **Backoff**: si un trigrama no se ha visto, caer a bigramas.
4. **`<EOS>` y `<BOS>`**: marcadores de inicio y fin de oracion.
5. **Redes neuronales**: pasar de contar a aprender representaciones.

Ya has construido un mini-modelo de lenguaje con mas contexto. Es el siguiente paso natural desde los bigramas hacia los modelos modernos.

In [ ]:
corpus_ampliado = """La inteligencia artificial es el futuro de la tecnologia.
La inteligencia artificial transforma el mundo cada dia.
La inteligencia artificial aprende de grandes cantidades de datos.
La inteligencia artificial procesa el lenguaje natural.
La inteligencia artificial genera texto e imagenes.
El futuro de la tecnologia es prometedor.
El futuro de la inteligencia artificial es brillante.
El futuro de la programacion es emocionante.
La tecnologia avanza rapidamente cada dia.
La tecnologia cambia la vida de las personas.
La tecnologia moderna depende de los datos.
La programacion en python es divertida y poderosa.
La programacion en python es sencilla y clara.
Python es un lenguaje muy popular.
Python permite crear aplicaciones rapidas.
Python procesa grandes cantidades de datos.
Los diccionarios en python son extremadamente rapidos.
Los diccionarios en python permiten busquedas rapidas.
Los diccionarios almacenan pares de clave y valor.
Las tablas hash permiten busquedas en tiempo constante.
Las tablas hash son la base de los diccionarios.
Las tablas hash almacenan datos de forma eficiente.
El lenguaje natural es dificil de procesar.
El procesamiento del lenguaje natural es un reto.
El procesamiento del lenguaje natural usa estadisticas.
El procesamiento del lenguaje natural usa modelos grandes.
Los modelos de lenguaje aprenden de grandes corpus.
Los modelos de lenguaje predicen la siguiente palabra.
Los modelos de lenguaje generan texto coherente.
Los modelos de lenguaje procesan millones de palabras.
La tokenizacion convierte palabras en numeros.
La tokenizacion es el primer paso del procesamiento.
La tokenizacion asigna identificadores a las palabras.
Las palabras desconocidas se marcan como desconocidas.
Las redes neuronales aprenden de los datos.
Las redes neuronales procesan grandes cantidades de informacion.
El aprendizaje automatico transforma industrias enteras.
El aprendizaje automatico usa grandes cantidades de datos.
Los datos son el combustible de la inteligencia artificial.
Los algoritmos procesan los datos rapidamente.
La ciencia de datos es fundamental hoy en dia.
La ciencia de datos combina estadistica y programacion.
El analisis de datos revela patrones ocultos.
El analisis de datos requiere herramientas poderosas.
Los patrones ocultos aparecen en los datos.
La estadistica es la base de los modelos.
La estadistica ayuda a entender los datos.
Los modelos estadisticos predicen el futuro.
Los modelos estadisticos procesan informacion compleja.
La informacion compleja requiere buenos algoritmos."""